# Team 3 — TTCC · URA Hackathon 2026 


## Cell 1 — Cài đặt

In [ ]:
import os
os.environ["FLAGS_use_mkldnn"]         = "0"
os.environ["FLAGS_enable_onednn"]      = "0"
os.environ["DNNL_DEFAULT_FPMATH_MODE"] = "STRICT"
os.environ["PADDLE_DISABLE_MKL"]       = "1"
os.environ["OMP_NUM_THREADS"]          = "2"

!pip install "paddlepaddle==2.6.2" -q
!pip install "paddleocr>=2.9,<3.0"  -q
!pip install tqdm scikit-learn Pillow rapidfuzz -q
!pip install vietocr==0.3.13 -q

# ── Patch tương thích Pillow mới (>=10) với paddleocr<3.0 ──
# paddleocr cũ gọi `from PIL._util import is_directory, is_path`,
# nhưng Pillow >=10 đã xoá 2 hàm này khỏi PIL/_util.py. Một số
# package khác (vd vietocr/gdown) có thể kéo theo Pillow mới đè
# lên bản cũ -> gây lỗi:
#   ImportError: cannot import name 'is_directory' from 'PIL._util'
# Patch lại 2 hàm này TRƯỚC khi import paddleocr để fix dứt điểm,
# không cần downgrade Pillow toàn hệ thống (tránh phá vỡ các thư
# viện khác trên Kaggle như ydata-profiling).
import PIL._util as _pil_util
if not hasattr(_pil_util, "is_directory"):
    _pil_util.is_directory = lambda f: os.path.isdir(f)
if not hasattr(_pil_util, "is_path"):
    _pil_util.is_path = lambda f: isinstance(f, (str, bytes, os.PathLike))

import paddle, paddleocr
from importlib.metadata import version as _pkg_version

print(f"paddlepaddle : {paddle.__version__}        (cần 2.6.2)")
print(f"paddleocr    : {_pkg_version('paddleocr')}  (cần <3.0)")
print(f"vietocr      : {_pkg_version('vietocr')}    (cần 0.3.13)")
import PIL
print(f"Pillow       : {PIL.__version__}  (patched for is_directory/is_path)")
print("Cài đặt xong ✓")



[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip
PLEASE USE OMP_NUM_THREADS WISELY.


paddlepaddle : 3.2.2        (cần 3.2.2)
paddleocr    : 2.10.0  (cần <3.0)
vietocr      : 0.3.13    (cần 0.3.13)
Pillow       : 10.2.0  (patched for is_directory/is_path)
Cài đặt xong ✓


## Cell 2 — Load Data

In [2]:
import re, time, zipfile, os
import pandas as pd
from pathlib import Path

try:
    from tqdm.notebook import tqdm
except ImportError:
    from tqdm import tqdm

from PIL import Image, ImageEnhance

# ══════════════════════════════════════════════════════════════
# TÌM ĐƯỜNG DẪN ĐỘNG (Tương thích mọi cấu trúc Phase 1 & 2)
# ══════════════════════════════════════════════════════════════
def locate_dataset():
    root = Path("/kaggle/input") if Path("/kaggle/input").exists() else Path(".")
    
    test_csv = None
    # 1. Tìm file CSV (ưu tiên Phase 2 - private_test.csv)
    for csv_name in ["private_test.csv", "test.csv"]:
        found = list(root.rglob(csv_name))
        if found:
            test_csv = found[0]
            break
            
    if not test_csv:
        raise FileNotFoundError("Không tìm thấy private_test.csv hoặc test.csv trong bộ dataset.")
        
    input_dir = test_csv.parent
    
    # 2. Tìm thư mục ảnh
    images_dir = None
    valid_exts = {".jpg", ".jpeg", ".png"}
    
    # Quét các thư mục xung quanh file CSV (chính nó và thư mục cha)
    for search_base in [input_dir, input_dir.parent]:
        if not search_base.exists(): continue
        for p in search_base.rglob("*"):
            if p.is_dir() and p.name in ["images", "test_images"]:
                # Nếu có ảnh bên trong thì chốt luôn
                if any(f.suffix.lower() in valid_exts for f in p.iterdir() if f.is_file()):
                    images_dir = p
                    break
        if images_dir:
            break
            
    # Fallback: Thử tìm zip để giải nén nếu không thấy thư mục
    if not images_dir:
        for zname in ["test_images.zip", "images.zip"]:
            for p in root.rglob(zname):
                er = Path("/kaggle/working/dataset_images")
                er.mkdir(exist_ok=True, parents=True)
                with zipfile.ZipFile(p) as zf:
                    zf.extractall(er)
                for ext_dir in er.rglob("*"):
                    if ext_dir.is_dir() and any(f.suffix.lower() in valid_exts for f in ext_dir.iterdir() if f.is_file()):
                        images_dir = ext_dir
                        break
                if images_dir: break
            if images_dir: break
            
    if not images_dir:
        raise FileNotFoundError(f"Đã tìm thấy {test_csv.name} nhưng KHÔNG tìm thấy thư mục ảnh chứa .jpg/.png.")
        
    # 3. Tìm train_labels.csv (Dù nó nằm ở root hay ở đâu cũng tìm ra)
    train_labels_csv = None
    found_labels = list(root.rglob("train_labels.csv"))
    if found_labels:
        train_labels_csv = found_labels[0]
        
    return input_dir, test_csv, images_dir, train_labels_csv

# --- INIT VARIABLES ---
INPUT_DIR, TEST_CSV, IMAGES_DIR, TRAIN_LABELS_CSV = locate_dataset()
WORK_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")

# Tự động match sample submission theo Phase
SAMPLE_CSV = INPUT_DIR / "sample_submission_private.csv" if (INPUT_DIR / "sample_submission_private.csv").exists() else INPUT_DIR / "sample_submission.csv"
OUTPUT_CSV = WORK_DIR / "submission.csv"
CHECKPOINT_CSV = WORK_DIR / "checkpoint.csv"

test_df = pd.read_csv(TEST_CSV)
# Load ID từ ổ đĩa (chấp nhận cả jpg, jpeg, png đề phòng Phase 2 dùng PNG)
on_disk = {p.stem for p in IMAGES_DIR.glob("*.*") if p.suffix.lower() in [".jpg", ".jpeg", ".png"]}

train_labels_df = None
if TRAIN_LABELS_CSV and TRAIN_LABELS_CSV.exists():
    train_labels_df = pd.read_csv(TRAIN_LABELS_CSV, keep_default_na=False)

print(f"Input dir   : {INPUT_DIR}")
print(f"Images dir  : {IMAGES_DIR}  ({len(on_disk):,} ảnh)")
print(f"Test file   : {TEST_CSV.name} ({len(test_df):,} images)")
print(f"Missing     : {len(set(test_df['image_id']) - on_disk)}")

if train_labels_df is not None:
    of = (train_labels_df["ocr_text"].astype(str).str.strip()!="").mean()
    pf = (train_labels_df["product_name"].astype(str).str.strip()!="").mean()
    print(f"Train labels: {len(train_labels_df):,} rows | OCR {of:.1%} | product {pf:.1%}")
else:
    print("Train labels: Không tìm thấy (Mô hình sẽ chạy hoàn toàn dựa trên rules)")

FileNotFoundError: Không tìm thấy private_test.csv hoặc test.csv trong bộ dataset.

## Cell 3 — Brand Rules: Brand_name & Product_name (tách riêng)


In [ ]:
import re
from unicodedata import normalize as _unorm

def _strip_accent(s: str) -> str:
    s = _unorm('NFD', s)
    s = re.sub(r'[\u0300-\u036f]', '', s)
    return _unorm('NFC', s.replace('đ','d').replace('Đ','D')).lower()

# ══════════════════════════════════════════════════════════════
# BRAND_RULES — (pattern, brand_canonical) theo priority cao→thấp
# Hạ Long family đặc biệt: brand tùy theo context (Canfoco vs công ty)
# ══════════════════════════════════════════════════════════════
BRAND_RULES = [
    # ── Hạ Long / Canfoco family ──
    (r'ha.?long.{0,25}canf[uo]c|canf[uo]c.{0,25}ha.?long|halongcanf',
     'Ha Long Canfoco'),
    (r'\bcanfoco\b|\bcanfood\b|canf[uo]co\b',
     'Ha Long Canfoco'),
    (r'(cong\s*ty|ctcp|co\s*phan|cty).{0,50}(do\s*hop|đồ\s*hộp).{0,30}ha.?long',
     'Đồ Hộp Hạ Long'),
    (r'(do\s*hop|đồ\s*hộp).{0,30}ha.?long',
     'Đồ Hộp Hạ Long'),
    (r'ha.?long.{0,30}(do\s*hop|đồ\s*hộp)',
     'Đồ Hộp Hạ Long'),
    (r'pate.{0,15}(c[oộ]t|cot).{0,10}(d[eèẻ]n|đèn)|patecot(?:den)?',
     'Pate Cột Đèn Hải Phòng'),
    (r'(c[oộ]t|cot).{0,10}(d[eèẻ]n|đèn).{0,20}(hai\s*phong|hải\s*phòng)',
     'Pate Cột Đèn Hải Phòng'),
    (r'ha.?long|halong',
     'Đồ Hộp Hạ Long'),

    # ── Nestlé NAN (sub-line đã gắn sẵn trong brand) ──
    (r'nan.{0,5}optipro.{0,5}plus|nan.{0,5}optiproplus',
     'Nestlé NAN OPTIpro PLUS'),
    (r'nan.{0,5}infinipro|nan.{0,5}infini\b',
     'Nestlé NAN INFINIPRO A2'),
    (r'nan.{0,5}supremepro|nan.{0,5}supreme',
     'Nestlé NAN SUPREMEpro'),
    (r'nan.{0,5}optipro\b',
     'Nestlé NAN OPTIpro'),
    (r'\bnan\b',
     'Nestlé NAN'),
    (r'\bbeba\b',
     'Nestlé BEBA'),
    (r'nestl[eé].{0,10}alfamino|\balfamino\b',
     'Nestlé Alfamino'),

    # ── Brand chính (có thể có product line riêng — xem LINE_RULES) ──
    (r'\bmilo\b',                  'Nestlé Milo'),
    (r'nestle|nestlé',               'Nestlé'),
    (r'vinamilk',                    'Vinamilk'),
    (r'th\s*true|thtrue',           'TH True Milk'),
    (r'dutch\s*lady',                'Dutch Lady'),
    (r'\baptamil\b',               'Aptamil'),
    (r'similac',                     'Abbott Similac'),
    (r'\bensure\b',                'Abbott Ensure'),
    (r'pediasure',                   'Abbott PediaSure'),
    (r'glucerna',                    'Abbott Glucerna'),
    (r'\bfriso\b',                 'Friso'),
    (r'\bhipp\b',                  'HiPP'),
    (r'nutifood|\bnuti\b',         'Nutifood'),
    (r'optimum\s*gold',             'Optimum Gold'),
    (r'\bmeiji\b',                 'Meiji'),
    (r'\billuma\b',                'Illuma'),
    (r'\banlene\b',                'Anlene'),
    (r'\byomost\b',                'Yomost'),
    (r'\bfami\b',                  'Fami'),
    (r'lothamilk',                   'Lothamilk'),
    (r'ba\s*v[iì]\b|\bbavi\b',    'Ba Vì'),
    (r'dalat\s*milk',               'Đà Lạt Milk'),
    (r'\bkun\b',                   'Kun'),

    # ── Coffee ──
    (r'highlands?.{0,5}coffee|\bhighlands\b', 'Highlands Coffee'),
    (r'the\s*coffee\s*house|coffee\s*house', 'Coffee House'),
    (r'phuc\s*long|phúc\s*long',              'Phúc Long'),

    # ── Canned / Pate khác ──
    (r'\bvissan\b',                'Vissan'),
    (r'chin[\s\-]*su|chinsu',       'Chin-Su'),
    (r'quang\s*h[oô]ng',            'Quang Hong Sardine'),
    (r'\bhafi\b',                  'Hafi'),
    (r'sardine|cá\s*mòi',           'Sardine'),
    (r'pate\s*minh\s*chay|minh\s*chay', 'Pate Minh Chay'),

    # ── Pate generic fallback (chỉ áp dụng nếu KHÔNG match brand nào ở trên) ──
    (r'\bpate\b|\bpatê\b',
     'Pate Cột Đèn Hải Phòng'),
]

# ══════════════════════════════════════════════════════════════
# LINE_RULES — product line riêng cho từng brand
# Brand đã chốt + tìm thêm line trong text → product_name = "Line"
# (Không gộp brand vào product_name nữa — BGK chấm 2 cột riêng)
# ══════════════════════════════════════════════════════════════
LINE_RULES = {
    'Ha Long Canfoco': [
        (r'pate.{0,10}(c[oộ]t|cot).{0,10}(d[eèẻ]n|đèn)|patecot', 'Pate Cột Đèn'),
    ],
    'Vinamilk': [
        (r'\bflex\b',            'Flex'),
        (r'adm\s*gold',           'ADM Gold'),
        (r'\bsure\b',            'Sure'),
        (r'colosbaby',             'ColosBaby'),
        (r'dielac',                'Dielac'),
        (r'\bgrow\b',            'Grow'),
        (r'\bcanxi\b',           'Canxi'),
        (r'ong\s*tho|ông\s*thọ', 'Ông Thọ'),
        (r'sua\s*chua|sữa\s*chua', 'Sữa Chua'),
    ],
    'Nestlé Milo': [
        (r'3\s*in\s*1|3in1',     '3in1'),
    ],
    'TH True Milk': [
        (r'true\s*yogurt',        'True Yogurt'),
        (r'\bgrow\b',            'Grow'),
        (r'school\s*milk',        'School Milk'),
        (r'true\s*butter',        'True Butter'),
    ],
    'Dutch Lady': [
        (r'grow\s*\+?|grow\s*plus', 'Grow+'),
        (r'complete',              'Complete'),
        (r'\bcanxi\b',           'Canxi'),
        (r'\bmom\b',             'Mum'),
    ],
    'Vissan': [
        (r'pate\s*heo',           'Pate Heo'),
        (r'pate\s*g[aà]\b',      'Pate Gà'),
        (r'xuc\s*xich|xúc\s*xích', 'Xúc Xích'),
        (r'cá\s*x[oố]t\s*c[aà]', 'Cá Xốt Cà'),
    ],
    'Highlands Coffee': [
        (r'tr[aà]\s*sen\s*v[aà]ng', 'Trà Sen Vàng'),
        (r'tr[aà]\s*v[aả]i',         'Trà Vải'),
        (r'americano',                'Americano'),
        (r'phuc\s*kien|phúc\s*kiến','Phúc Kiến'),
    ],
    'Nutifood': [
        (r'growplus|grow\s*plus', 'GrowPlus'),
        (r'\bpedia\b',           'Pedia'),
    ],
    'Aptamil': [
        (r'profutura',             'Profutura'),
    ],
    'Friso': [
        (r'\bgold\b',            'Gold'),
        (r'comfort',               'Comfort'),
        (r'prestige',              'Prestige'),
    ],
    'Abbott Ensure': [
        (r'\bgold\b',            'Gold'),
    ],
    'Anlene': [
        (r'\bgold\b',            'Gold'),
        (r'concentrate',           'Concentrate'),
    ],
    'Ba Vì': [
        (r'\bgold\b',            'Gold'),
    ],
    'Kun': [
        (r'chocolate',             'Chocolate'),
        (r'strawberry',            'Strawberry'),
    ],
}

# Brand nào đã có sub-line gắn sẵn trong tên brand (NAN OPTIpro PLUS,
# Pate Cột Đèn Hải Phòng, Đồ Hộp Hạ Long...) → KHÔNG tìm line riêng nữa,
# product_name sẽ rỗng cho các brand này trừ khi nằm trong LINE_RULES.
_BRAND_HAS_BUILTIN_LINE = {
    'Nestlé NAN OPTIpro PLUS', 'Nestlé NAN INFINIPRO A2',
    'Nestlé NAN SUPREMEpro', 'Nestlé NAN OPTIpro',
    'Pate Cột Đèn Hải Phòng', 'Đồ Hộp Hạ Long',
}

# Sản phẩm/text noise — không phải brand/product hợp lệ
NOISE_PRODUCTS = {
    'NS RECORDS', 'Bình trữ sữa', 'CapCut', 'TikTok',
    'Finance', 'ChatGPT', 'LINH IT', 'CP',
}

# Chuẩn hoá BRAND (không còn chuẩn hoá brand+line gộp như trước)
BRAND_NORMALIZE_MAP = [
    (r'^ha long canfood$',               'Ha Long Canfoco'),
    (r'^nan$|^sữa nan$',                 'Nestlé NAN'),
    (r'^nestle beba$',                   'Nestlé BEBA'),
]


def _find_brand(tl: str, tls: str) -> str:
    for pattern, brand in BRAND_RULES:
        if re.search(pattern, tl, re.IGNORECASE) or re.search(pattern, tls, re.IGNORECASE):
            return brand
    return ''


def _find_line(brand: str, tl: str, tls: str) -> str:
    for pattern, line in LINE_RULES.get(brand, []):
        if re.search(pattern, tl, re.IGNORECASE) or re.search(pattern, tls, re.IGNORECASE):
            return line
    return ''


def extract_brand_product(text: str) -> tuple:
    """Trả về (brand_name, product_name) riêng biệt theo spec BGK mới.

    brand_name   : tên hãng/brand canonical (VD: 'Vinamilk', 'Highlands Coffee')
    product_name : dòng sản phẩm / line CHỈ — không gộp brand vào
                   (VD: 'Flex', 'Trà Sen Vàng'); rỗng nếu không match line nào.
    """
    if not text or not text.strip():
        return '', ''

    # Mask "Công ty CP ..." → 'CP' không bị match như brand riêng
    clean = re.sub(
        r'(công\s*ty|cty|ctcp|cổ\s*phần|tổng\s*giám\s*đốc|giám\s*đốc)\s*cp\b',
        'COTYCP', text, flags=re.IGNORECASE
    )
    tl  = clean.lower()
    tls = _strip_accent(clean)

    brand = _find_brand(tl, tls)
    if not brand:
        return '', ''

    line = ''
    if brand not in _BRAND_HAS_BUILTIN_LINE:
        line = _find_line(brand, tl, tls)

    return brand, line


def normalize_brand(name: str) -> str:
    if not name: return ''
    name = name.strip()
    for pat, canon in BRAND_NORMALIZE_MAP:
        if re.fullmatch(pat, name, re.IGNORECASE):
            return canon
    return name


def normalize_product(name: str) -> str:
    if not name: return ''
    name = name.strip()
    if name in NOISE_PRODUCTS: return ''
    # Loại bỏ giá/khối lượng/hashtag còn sót (phòng hờ)
    name = re.sub(r'#\S+', '', name).strip()
    name = re.sub(r'\b\d+[.,]?\d*\s*(g|kg|ml|l|đ|%|k)\b', '', name, flags=re.IGNORECASE).strip()
    if len(name) > 60: return ''
    return name


def predict_brand_product(ocr_text: str) -> tuple:
    """Pipeline rule-based đơn (không qua ML). Dùng làm tầng 1."""
    brand, line = extract_brand_product(ocr_text)
    return normalize_brand(brand), normalize_product(line)


# Self-test
tests = [
    ("HALONG CANFOCO PATE CỘT ĐÈN HẢI PHÒNG",     ('Ha Long Canfoco', 'Pate Cột Đèn')),
    ("Công TY CP Đồ hộp Hạ Long bị bắt",         ('Đồ Hộp Hạ Long', '')),
    ("DO HOP HA LONG ISO 22000",                   ('Đồ Hộp Hạ Long', '')),
    ("PATE COT DEN CUA DO HOP HA LONG",            ('Đồ Hộp Hạ Long', '')),
    ("Sữa tươi tiệt trùng Vinamilk Flex 180ml",    ('Vinamilk', 'Flex')),
    ("Vinamilk EST 1976 thông báo khẩn",           ('Vinamilk', '')),
    ("NESTLÉ MILO Chocolate Malt Drink 3in1",      ('Nestlé Milo', '3in1')),
    ("Vissan PATE HEO 170g combo 3 hộp",           ('Vissan', 'Pate Heo')),
    ("Dutch Lady Grow+ 900g",                      ('Dutch Lady', 'Grow+')),
    ("Ba Vì Gold 1L",                              ('Ba Vì', 'Gold')),
    ("NAN OPTIPROPLUS thu hồi sữa",                ('Nestlé NAN OPTIpro PLUS', '')),
    ("LỜI NHẮN NHỦ VỀ NHÂN QUẢ VÀ DI SẢN",        ('', '')),
    ("b3 ifffÙ 200v set ị PIN",                    ('', '')),
    ("LS",                                          ('', '')),
]
passed = 0
print("Brand/Product split rules self-test:")
for text, exp in tests:
    got = predict_brand_product(text)
    ok  = got == exp
    passed += ok
    print(f"  {'✅' if ok else '❌'} '{text[:48]}' → {got}" + ('' if ok else f"  (exp: {exp})"))
print(f"  {passed}/{len(tests)} passed | Brand rules: {len(BRAND_RULES)} | Line rules: {sum(len(v) for v in LINE_RULES.values())}")


## Cell 3b — TF-IDF + Discovery Layer (heuristic+NER) + Online Cache cho Brand_name & Product_name


In [ ]:
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
import time

try:
    from rapidfuzz import process, fuzz
    _HAS_RAPIDFUZZ = True
except ImportError:
    _HAS_RAPIDFUZZ = False
    print("⚠️ rapidfuzz chưa cài — fuzzy fallback sẽ bị bỏ qua. Cài bằng: pip install rapidfuzz")


# ══════════════════════════════════════════════════════════════
# Train labels CŨ chỉ có 1 cột product_name (gộp Brand [+ Line]).
# Suy ra (brand, line) riêng để fit 2 classifier độc lập:
#   - brand: derive từ ocr_text bằng rule (nguồn tin cậy nhất, có full
#     context câu) — KHÔNG derive từ product_name gộp vì có thể match
#     sai brand (vd "PATE CỘT ĐÈN HẢI PHÒNG" tự nó là 1 brand riêng,
#     không phải lúc nào cũng là Ha Long Canfoco).
#   - line: tìm trong (ocr_text + product_name gộp) bằng LINE_RULES,
#     chỉ áp dụng cho brand KHÔNG có sub-line gắn sẵn.
# ══════════════════════════════════════════════════════════════
def _derive_brand_line(row):
    brand, _ = extract_brand_product(row["ocr_text"])
    line = ""
    if brand and brand not in _BRAND_HAS_BUILTIN_LINE:
        combined = f'{row["ocr_text"]} {row["product_name"]}'
        line = _find_line(brand, combined.lower(), _strip_accent(combined))
    return pd.Series({"brand_derived": brand, "line_derived": line})


class NamePredictor:
    """2 tầng: (1) classifier nhị phân có/không có nhãn, (2) classifier
    đa lớp dự đoán tên cụ thể (brand hoặc line) — dùng chung kiến trúc
    cho cả brand_name và product_name(line)."""

    def __init__(self, prob_threshold=0.45, max_features=15000, min_class_count=2):
        self.prob_threshold  = prob_threshold
        self.max_features    = max_features
        self.min_class_count = min_class_count
        self._has_clf = self._name_clf = None

    def fit(self, texts, labels):
        texts  = pd.Series(texts).astype(str).str.strip()
        labels = pd.Series(labels).astype(str).str.strip()
        tf = dict(analyzer="char_wb", ngram_range=(2,4),
                  max_features=self.max_features, min_df=2, sublinear_tf=True)
        lr = dict(max_iter=500, class_weight="balanced", C=2.0)

        self._has_clf = Pipeline([("t", TfidfVectorizer(**tf)),
                                   ("c", LogisticRegression(**lr))])
        self._has_clf.fit(texts, (labels != "").astype(int))

        pos  = pd.DataFrame({"text": texts, "label": labels})
        pos  = pos[(pos["text"] != "") & (pos["label"] != "")]
        keep = pos["label"].value_counts()
        pos  = pos[pos["label"].isin(keep[keep >= self.min_class_count].index)]

        self._name_clf = None
        if len(pos) and pos["label"].nunique() >= 2:
            self._name_clf = Pipeline([("t", TfidfVectorizer(**tf)),
                                        ("c", LogisticRegression(**lr))])
            self._name_clf.fit(pos["text"], pos["label"])
        return self

    def predict(self, text: str) -> str:
        text = "" if text is None else str(text).strip()
        if not text or not self._has_clf: return ""
        proba   = self._has_clf.predict_proba([text])[0]
        classes = list(self._has_clf.classes_)
        if 1 not in classes or proba[classes.index(1)] < self.prob_threshold: return ""
        if not self._name_clf: return ""
        return str(self._name_clf.predict([text])[0])


# ── Fuzzy fallback: vớt các case OCR lỗi nặng mà rule + ML đều miss ──
_KNOWN_BRANDS = sorted(set(b for _, b in BRAND_RULES))
_KNOWN_LINES  = sorted(set(l for rules in LINE_RULES.values() for _, l in rules))

def fuzzy_match_fallback(text: str, candidates, score_cutoff: int = 82) -> str:
    """Fallback cuối cùng: so khớp gần đúng (typo-tolerant)."""
    if not _HAS_RAPIDFUZZ or not text or not text.strip() or not candidates:
        return ""
    match = process.extractOne(
        text, candidates, scorer=fuzz.partial_ratio, score_cutoff=score_cutoff
    )
    return match[0] if match else ""


brand_predictor = None
line_predictor  = None

if train_labels_df is not None:
    tl_df = train_labels_df.copy()
    tl_df["ocr_text"]     = tl_df["ocr_text"].astype(str).str.strip()
    tl_df["product_name"] = tl_df["product_name"].astype(str).str.strip()

    derived = tl_df.apply(_derive_brand_line, axis=1)
    tl_df   = pd.concat([tl_df, derived], axis=1)

    brand_predictor = NamePredictor()
    brand_predictor.fit(tl_df["ocr_text"], tl_df["brand_derived"])

    line_predictor = NamePredictor(min_class_count=2)
    line_predictor.fit(tl_df["ocr_text"], tl_df["line_derived"])

    n_brand = (tl_df["brand_derived"] != "").sum()
    n_line  = (tl_df["line_derived"]  != "").sum()
    print(f"Trained ✓  |  brand labels: {n_brand:,}  |  line labels: {n_line:,}")
else:
    print("Không có train_labels — chỉ dùng rules")


def predict_brand_name(ocr_text: str) -> str:
    """3 tầng cho brand_name: rule (precision cao nhất) → ML → fuzzy fallback."""
    brand, _ = extract_brand_product(ocr_text)
    if not brand and brand_predictor is not None:
        brand = brand_predictor.predict(ocr_text)
    if not brand:
        brand = fuzzy_match_fallback(ocr_text, _KNOWN_BRANDS)
    return normalize_brand(brand)


def predict_product_name(ocr_text: str, brand: str = "") -> str:
    """3 tầng cho product_name (line): rule → ML → fuzzy fallback.
    Brand đã có sub-line gắn sẵn (NAN OPTIpro..., Pate Cột Đèn HP,
    Đồ Hộp Hạ Long) → product_name luôn rỗng, không cố tìm thêm."""
    if brand in _BRAND_HAS_BUILTIN_LINE:
        return ""
    _, line = extract_brand_product(ocr_text)
    if not line and line_predictor is not None:
        line = line_predictor.predict(ocr_text)
    if not line and brand:
        line = fuzzy_match_fallback(ocr_text, _KNOWN_LINES)
    return normalize_product(line)


# ══════════════════════════════════════════════════════════════
# TẦNG 4 — DISCOVERY LAYER (cho brand/product CHƯA TỪNG biết)
# Chạy SAU rule (tầng 1) + ML (tầng 2) + fuzzy (tầng 3) đều miss.
# Kết hợp 2 chiến lược, lấy ứng viên có confidence cao hơn:
#   (a) Heuristic thương hiệu: viết hoa liên tiếp, ký hiệu TM/®/©,
#       vị trí đầu câu, độ dài cụm hợp lý.
#   (b) "NER-lite" bằng POS-pattern: do không có model NER tiếng
#       Việt nhẹ chạy offline trên Kaggle, ta xấp xỉ NER bằng cách
#       tìm cụm liên tiếp các từ "trông giống danh từ riêng" (mỗi
#       từ đều viết hoa chữ đầu, không phải stopword) — đúng kiểu
#       pattern NER rule-based cổ điển (capitalization-based NER).
# ══════════════════════════════════════════════════════════════

# Stopword/blacklist: từ không bao giờ là brand/product dù viết hoa
_DISCOVERY_STOPWORDS = {
    # tiếng Việt thông dụng trong tin tức/MXH — không phải brand
    'tin', 'tức', 'tin tức', 'báo', 'mới', 'nóng', 'khẩn', 'sở', 'y', 'tế',
    'công', 'ty', 'cổ', 'phần', 'ctcp', 'cty', 'việt', 'nam', 'hà', 'nội',
    'sài', 'gòn', 'thành', 'phố', 'quận', 'huyện', 'tỉnh', 'review',
    'mukbang', 'shorts', 'video', 'kênh', 'channel', 'cập', 'nhật',
    'thông', 'báo', 'cảnh', 'giác', 'chú', 'ý', 'lưu', 'người', 'dân',
    'theo', 'dõi', 'chia', 'sẻ', 'group', 'fanpage', 'admin', 'official',
    'tiktok', 'facebook', 'youtube', 'instagram', 'capcut', 'content',
    'news', 'breaking', 'live', 'trực', 'tiếp', 'hôm', 'nay', 'ngày',
    'tháng', 'năm', 'sáng', 'chiều', 'tối', 'đêm', 'giờ', 'phút',
    # từ hành động/tính từ thường gặp trong tiêu đề clickbait viết hoa
    # toàn bộ (KINH HOÀNG, HÚT, SỐC, NÓNG...) — không phải brand
    'hút', 'sốc', 'kinh', 'hoàng', 'gây', 'rúng', 'động', 'phẫn', 'nộ',
    'nguy', 'hiểm', 'cấp', 'khủng', 'khiếp',
    'bất', 'ngờ', 'bùng', 'nổ', 'lan', 'truyền', 'viral', 'hot',
    # địa danh phổ biến — không phải brand dù viết hoa đầu câu
    'singapore', 'thái', 'lan', 'trung', 'quốc', 'nhật', 'bản', 'hàn',
    'mỹ', 'pháp', 'đức', 'anh', 'úc', 'malaysia', 'indonesia',
    'philippines', 'lào', 'campuchia', 'châu', 'á', 'âu', 'phi',
    'đà', 'nẵng', 'huế', 'cần', 'thơ', 'hải', 'phòng', 'biên', 'hòa',
    # giới từ/liên từ/đại từ tiếng Việt phổ biến — không bao giờ là
    # brand dù bị OCR/viết tiêu đề ra toàn chữ hoa ("HÚT TRONG ĐÊM")
    'trong', 'ngoài', 'trên', 'dưới', 'và', 'hoặc', 'nhưng', 'là',
    'của', 'cho', 'với', 'từ', 'đến', 'khi', 'nếu', 'vì', 'do', 'bởi',
    'này', 'đó', 'kia', 'ấy', 'đây', 'đấy', 'rồi', 'đã', 'sẽ', 'đang',
    'không', 'chưa', 'còn', 'cũng', 'chỉ', 'rất', 'quá', 'lắm',
    # đại từ nhân xưng + từ phổ thông hay đứng đầu câu tiếng Việt
    # (false positive thường gặp: "Thu", "Bạn", "Cụ", "Buông", "Hàng")
    'tôi', 'bạn', 'chúng', 'họ', 'nó', 'mình', 'ta', 'cụ', 'ông', 'bà',
    'anh', 'chị', 'em', 'cô', 'thu', 'chi', 'buông', 'hàng', 'đoàn',
    'kết', 'cùng', 'phải', 'không', 'cuộc', 'đời', 'tương', 'lai',
    'vượt', 'khó', 'trao', 'hi', 'vọng', 'tạo',
}

# Domain whitelist hints — tăng confidence nếu cụm từ đứng gần các
# từ khoá liên quan FMCG/thực phẩm (tăng recall đúng hướng bài toán)
# Domain hint — chỉ giữ từ khoá ngành hàng THẬT SỰ đặc trưng, đủ dài
# để không trùng tình cờ. Bỏ các đơn vị đo siêu ngắn (g, ml, kg...)
# và từ chung (nước, bánh, kẹo, hộp, lon...) vì chúng quá phổ biến
# trong MỌI loại câu, khiến domain-hint gate dễ bị pass nhầm.
_DOMAIN_HINT_WORDS = {
    'sữa', 'milk', 'cà phê', 'coffee', 'trà', 'tea', 'pate', 'patê',
    'đồ hộp', 'xúc xích', 'gia vị', 'bánh kẹo', 'nước giải khát',
    'thu hồi', 'sản phẩm', 'nhãn hàng', 'thương hiệu', 'dầu ăn',
    'thực phẩm', 'tiệt trùng', 'công thức', 'dinh dưỡng', 'hộp sữa',
}

# Tín hiệu NGỮ CẢNH THƯƠNG MẠI tổng quát — không cần đúng tên ngành
# hàng cụ thể (vì ngành hàng mới có thể chưa nằm trong whitelist
# trên), nhưng câu có các cụm này thường đang nói về MỘT THƯƠNG
# HIỆU/SẢN PHẨM cụ thể (ra mắt, giảm giá, lỗi, khủng hoảng truyền
# thông...) hơn là tin tức/MXH chung. Dùng làm lối đi THỨ HAI qua
# gate, bên cạnh domain hint theo ngành hàng cụ thể.
_COMMERCIAL_CONTEXT_WORDS = {
    'ra mắt', 'giảm giá', 'khuyến mãi', 'ưu đãi', 'mua ngay',
    'chính thức có mặt', 'phân phối', 'nhập khẩu', 'chính hãng',
    'lô hàng', 'lỗi', 'khủng hoảng truyền thông', 'tẩy chay',
    'tăng giá', 'mở bán', 'ra đời', 'bộ sưu tập', 'dòng sản phẩm',
}

_TM_SYMBOLS = re.compile(r'[™®©]')
_VN_DIACRITIC_RE = re.compile(r'[\u00C0-\u024F\u1EA0-\u1EF9]')


def _tokenize_with_spans(text: str):
    """Tách từ giữ nguyên hoa/thường, kèm vị trí (start, end) gốc
    trong text — cần để định vị chính xác run khi ghép token lại
    (text.find() có thể fail nếu giữa các token có ký tự bị loại
    bỏ như '&', dấu câu, nhiều khoảng trắng...)."""
    return [(m.group(0), m.start(), m.end())
            for m in re.finditer(r"[A-Za-zÀ-ỹ0-9]+(?:[''\-][A-Za-zÀ-ỹ]+)*", text)]


def _looks_like_proper_noun(tok: str) -> bool:
    """True nếu token 'trông giống' tên riêng: chữ đầu hoa, không phải
    toàn bộ là số, không phải stopword."""
    if not tok or tok.lower() in _DISCOVERY_STOPWORDS: return False
    if tok.isdigit(): return False
    if len(tok) < 2: return False
    return tok[0].isupper()


def _extract_capitalized_runs(tokens_with_spans):
    """NER-lite: tìm các 'run' liên tiếp gồm toàn token trông giống
    danh từ riêng (capitalization-based NER pattern cổ điển).
    Trả về list (run_text, start_pos, end_pos) — start_pos/end_pos
    là vị trí THẬT trong text gốc (token đầu/cuối của run), không
    phải vị trí sau khi ghép lại bằng ' '.join (có thể lệch nếu
    giữa các từ có ký tự đặc biệt như '&', dấu phẩy...)."""
    runs, cur = [], []
    for tok, s, e in tokens_with_spans:
        if _looks_like_proper_noun(tok):
            cur.append((tok, s, e))
        else:
            if len(cur) >= 1: runs.append(cur)
            cur = []
    if len(cur) >= 1: runs.append(cur)

    out = []
    for r in runs:
        if not (1 <= len(r) <= 4): continue
        run_text = ' '.join(t for t, _, _ in r)
        start_pos, end_pos = r[0][1], r[-1][2]
        out.append((run_text, start_pos, end_pos))
    return out


def _score_candidate(cand: str, text: str, start_pos: int) -> float:
    """Chấm confidence (0-1) cho 1 ứng viên brand/product dựa trên
    các tín hiệu heuristic độc lập, cộng dồn có trọng số."""
    score = 0.0
    toks = cand.split()
    all_caps = all(t.isupper() for t in toks if len(t) > 1)

    window = text[max(0, start_pos-30): start_pos+len(cand)+30].lower()
    has_domain_hint = (any(hint in window for hint in _DOMAIN_HINT_WORDS)
                       or any(hint in window for hint in _COMMERCIAL_CONTEXT_WORDS))

    # (1) Tất cả từ đều viết hoa chữ đầu hoặc toàn hoa -> +0.25
    if all(t[0].isupper() for t in toks if t):
        score += 0.25

    # (2) Có ký hiệu TM/®/© ngay sau cụm -> tín hiệu thương hiệu rất mạnh
    tail = text[start_pos + len(cand): start_pos + len(cand) + 3]
    if _TM_SYMBOLS.search(tail):
        score += 0.35

    # (3) Đứng ở đầu câu/đầu text -> +0.10 (giảm trọng số so với trước,
    #     vì clickbait/địa danh cũng thường ở đầu câu -> tín hiệu yếu
    #     hơn domain hint, không nên lấn lướt nó)
    if start_pos <= 2:
        score += 0.10

    # (4) Gần từ khoá ngành hàng (sữa, pate, đồ hộp...) trong cùng câu
    #     -> tín hiệu QUAN TRỌNG NHẤT cho đúng domain bài toán
    if has_domain_hint:
        score += 0.30

    # (5) Độ dài hợp lý (2-25 ký tự, 1-3 từ) -> brand/product thật
    #     thường ngắn gọn, cụm quá dài có thể là tên bài báo/câu văn
    if 2 <= len(cand) <= 25 and 1 <= len(toks) <= 3:
        score += 0.10
    else:
        score -= 0.15

    # (6) Run TOÀN HOA nhiều từ (≥2) là đặc trưng tiêu đề clickbait
    #     ("KINH HOÀNG", "HÚT TRONG ĐÊM") nhiều hơn là brand thật.
    #     Brand thật toàn hoa thường CHỈ 1 từ (vd "VINAMILK"). Chỉ
    #     tin tưởng run toàn-hoa ≥2 từ khi có domain hint đi kèm.
    if all_caps and len(toks) >= 2 and not has_domain_hint:
        score -= 0.40

    # (7) Brand/product trong dataset này (FMCG VN) thường là tên
    #     KHÔNG dấu (Vinamilk, Dumex, Alula, Heinz, Nestle...) vì
    #     phần lớn là thương hiệu quốc tế hoặc tên riêng phiên âm.
    #     Cụm 1 từ CÓ dấu tiếng Việt (vd "Bạn", "Cụ", "Buông") gần
    #     như luôn là từ tiếng Việt thông thường, không phải brand
    #     -> phạt nhẹ để giảm false positive, nhưng không loại hẳn
    #     (vẫn có brand Việt thật như "Ba Vì", "Đà Lạt Milk").
    if len(toks) == 1 and _VN_DIACRITIC_RE.search(cand) and not has_domain_hint:
        score -= 0.20

    return max(0.0, min(1.0, score))


def discover_brand_product(ocr_text: str, min_confidence: float = 0.45) -> tuple:
    """Tầng 4 (fallback cuối): khi rule + ML + fuzzy đều miss, thử
    'phát hiện' brand/product mới bằng heuristic + NER-lite.

    Trả về (brand_guess, product_guess, confidence) — guess chỉ
    được dùng khi confidence >= min_confidence, để tránh đưa rác
    vào submission khi không thật sự tự tin.
    """
    if not ocr_text or not ocr_text.strip():
        return '', '', 0.0

    tokens = _tokenize_with_spans(ocr_text)
    if not tokens:
        return '', '', 0.0

    runs = _extract_capitalized_runs(tokens)
    if not runs:
        return '', '', 0.0

    candidates = []
    for run_text, start_pos, end_pos in runs:
        sc = _score_candidate(run_text, ocr_text, start_pos)
        candidates.append((run_text, sc, start_pos))

    if not candidates:
        return '', '', 0.0

    # Sắp theo confidence giảm dần, ưu tiên xuất hiện sớm trong câu
    candidates.sort(key=lambda x: (-x[1], x[2]))

    best_brand = candidates[0]

    # GATE CỨNG: vì text thực tế trong dataset này lẫn rất nhiều tin
    # tức/MXH (vd "BÁC SĨ CÀ KHỊA" tên 1 kênh content, "KHỞI TỐ BÁC
    # PHẠM MINH CHÍNH"...), CHỈ heuristic "viết hoa + đầu câu" là
    # KHÔNG đủ để tin một cụm là brand sản phẩm. Bắt buộc phải có
    # domain hint (gần từ khoá ngành hàng: sữa, pate, đồ hộp...)
    # HOẶC ký hiệu TM/®/© rõ ràng, nếu không thì bỏ qua dù confidence
    # tính được có vẻ cao — đánh đổi giảm recall để tăng precision,
    # cần thiết vì noise thực tế nhiều hơn dự kiến ban đầu.
    window = ocr_text[max(0, best_brand[2]-30): best_brand[2]+len(best_brand[0])+30].lower()
    has_strong_signal = (
        any(hint in window for hint in _DOMAIN_HINT_WORDS)
        or any(hint in window for hint in _COMMERCIAL_CONTEXT_WORDS)
        or bool(_TM_SYMBOLS.search(ocr_text[best_brand[2]+len(best_brand[0]): best_brand[2]+len(best_brand[0])+3]))
    )
    if not has_strong_signal:
        return '', '', best_brand[1]

    if best_brand[1] < min_confidence:
        return '', '', best_brand[1]

    brand_guess = best_brand[0]
    # Ứng viên thứ 2 (nếu có, không trùng brand) -> coi là product guess
    product_guess = ''
    for cand, sc, pos in candidates[1:]:
        if cand.lower() != brand_guess.lower() and sc >= min_confidence * 0.7:
            product_guess = cand
            break

    return brand_guess, product_guess, round(best_brand[1], 3)


def predict_brand_product(ocr_text: str) -> tuple:
    """Hàm tổng hợp 4 TẦNG: rule → ML → fuzzy → discovery (heuristic+NER-lite).
    Mỗi tầng chỉ chạy nếu tầng trước chưa cho ra brand."""
    brand = predict_brand_name(ocr_text)
    prod  = predict_product_name(ocr_text, brand)

    if not brand:
        d_brand, d_prod, conf = discover_brand_product(ocr_text)
        if d_brand:
            brand = d_brand
            if not prod:
                prod = d_prod
    return brand, prod


# ══════════════════════════════════════════════════════════════
# TẦNG 5 — ONLINE SELF-LEARNING CACHE
# Không train lại model. Khi Discovery Layer (tầng 4) đoán ra 1
# brand MỚI đủ tin cậy, brand đó được ghi vào cache (RAM, sống
# suốt quá trình chạy Cell 5 — Main Loop). Các ảnh xử lý SAU
# trong cùng batch mà Discovery Layer đoán ra brand TRÙNG (khớp
# chính xác, không phân biệt hoa/thường) với brand đã có trong
# cache sẽ:
#   (a) Dùng lại đúng chính tả `canonical` đã lưu lần đầu, để
#       cùng 1 brand không bị ghi 2 kiểu khác nhau giữa các ảnh
#       (vd ảnh 1 ra "Alula", ảnh 50 ra "ALULA" -> đều thành "Alula").
#   (b) KHÔNG cần đạt min_confidence nữa — vì đã được "xác nhận"
#       bởi lần xuất hiện trước, ngưỡng tin tưởng tăng theo số
#       lần thấy lại (confirmed_count).
# Cache CHỈ khớp chính xác theo brand_lower — không tự gộp các
# biến thể gần giống (vd "Alula" và "Alula Gold" là 2 entry riêng)
# để tránh gộp nhầm 2 brand tình cờ giống tên.
# ══════════════════════════════════════════════════════════════

_discovery_cache: dict = {}   # brand_lower -> {"canonical", "count", "products": Counter}

def reset_discovery_cache():
    """Gọi hàm này khi muốn bắt đầu batch mới hoàn toàn từ đầu
    (cache không tự reset giữa các lần chạy Cell 5)."""
    global _discovery_cache
    _discovery_cache = {}
    print("Discovery cache đã reset (0 brand đã học).")


def _cache_lookup(brand_guess: str):
    """Tra cache theo khớp CHÍNH XÁC (case-insensitive). Trả về
    canonical đã lưu nếu có, None nếu chưa từng thấy."""
    if not brand_guess: return None
    entry = _discovery_cache.get(brand_guess.strip().lower())
    return entry["canonical"] if entry else None


def _cache_learn(brand_guess: str, product_guess: str = ""):
    """Ghi/nhật brand mới vào cache. Nếu đã có, chỉ tăng count +
    ghi nhận product_guess đi kèm (để biết product nào phổ biến
    nhất với brand này, dùng làm gợi ý phụ)."""
    if not brand_guess: return
    key = brand_guess.strip().lower()
    if key not in _discovery_cache:
        _discovery_cache[key] = {
            "canonical": brand_guess.strip(),
            "count": 0,
            "products": Counter(),
        }
    _discovery_cache[key]["count"] += 1
    if product_guess:
        _discovery_cache[key]["products"][product_guess.strip()] += 1


def _cache_search_in_text(ocr_text: str):
    """Tìm trực tiếp xem có brand NÀO trong cache xuất hiện dạng
    substring (khớp từ, không phân biệt hoa/thường) trong ocr_text.
    Cần thiết vì _extract_capitalized_runs() chỉ bắt được cụm có
    chữ HOA đầu — khi OCR yếu ra toàn chữ thường (vd 'alula gold
    reflux'), candidate đó sẽ không được tạo ra để so khớp cache
    theo đường thông thường. Tìm trực tiếp bằng regex word-boundary
    đảm bảo cache vẫn vớt được các trường hợp này."""
    if not ocr_text or not _discovery_cache: return None
    tl = ocr_text.lower()
    # Ưu tiên brand DÀI hơn trước (tránh match nhầm brand ngắn là
    # substring của brand dài, vd "Alula" trong "Alula Gold")
    for key in sorted(_discovery_cache.keys(), key=len, reverse=True):
        if re.search(rf'\b{re.escape(key)}\b', tl):
            return _discovery_cache[key]["canonical"]
    return None


def discover_with_cache(ocr_text: str, min_confidence: float = 0.45) -> tuple:
    """Wrapper quanh discover_brand_product() có thêm tầng cache:
      1) Chạy discover_brand_product() như cũ để lấy ứng viên thô.
      2) Nếu ứng viên KHỚP brand đã có trong cache -> dùng canonical
         đã lưu, BỎ điều kiện min_confidence (đã được xác nhận rồi).
      3) Nếu ứng viên MỚI và đạt min_confidence -> học vào cache.
      4) Nếu ứng viên MỚI nhưng KHÔNG đạt min_confidence -> thử tìm
         trực tiếp brand đã có trong cache dạng substring trong text
         (vớt lại case OCR yếu/viết thường mà NER-lite không bắt
         được dạng cụm viết hoa, nhưng brand đó đã từng xác nhận).
    """
    brand_guess, product_guess, conf = discover_brand_product(ocr_text, min_confidence=0.0)

    if brand_guess:
        cached_canonical = _cache_lookup(brand_guess)
        if cached_canonical is not None:
            _cache_learn(brand_guess, product_guess)
            return cached_canonical, product_guess, max(conf, min_confidence)

        if conf >= min_confidence:
            _cache_learn(brand_guess, product_guess)
            return brand_guess, product_guess, conf

    # Không có candidate đủ tốt từ NER-lite -> thử tìm trực tiếp
    # brand đã biết (trong cache) dạng substring, không cần viết hoa
    direct_hit = _cache_search_in_text(ocr_text)
    if direct_hit is not None:
        _cache_learn(direct_hit)
        return direct_hit, '', min_confidence

    return '', '', conf


def print_discovery_cache_summary(min_count: int = 2):
    """In tóm tắt các brand đã được cache học được trong batch
    hiện tại — tiện theo dõi cache đang hoạt động ra sao."""
    if not _discovery_cache:
        print("Cache rỗng — chưa học được brand mới nào.")
        return
    items = sorted(_discovery_cache.items(), key=lambda kv: -kv[1]["count"])
    print(f"Discovery cache: {len(items)} brand đã học trong batch này.")
    shown = [it for it in items if it[1]["count"] >= min_count]
    print(f"  (hiển thị {len(shown)} brand xuất hiện ≥{min_count} lần)\n")
    for key, info in shown:
        top_products = ', '.join(p for p, _ in info["products"].most_common(3)) or '(không có)'
        print(f"  '{info['canonical']}'  (x{info['count']})  product phổ biến: {top_products}")


def predict_brand_product(ocr_text: str) -> tuple:
    """Hàm tổng hợp 5 TẦNG: rule → ML → fuzzy → discovery (heuristic+
    NER-lite) → online cache (brand mới học được trong batch này).
    Mỗi tầng chỉ chạy nếu tầng trước chưa cho ra brand."""
    brand = predict_brand_name(ocr_text)
    prod  = predict_product_name(ocr_text, brand)

    if not brand:
        d_brand, d_prod, conf = discover_with_cache(ocr_text)
        if d_brand:
            brand = d_brand
            if not prod:
                prod = d_prod
    return brand, prod


if train_labels_df is not None:
    t0 = time.perf_counter()
    for _ in range(200): predict_brand_product("Vinamilk Flex 180ml")
    print(f"Inference: {(time.perf_counter()-t0)*1000/200:.3f} ms/img")

print("\nDiscovery Layer (tầng 4) self-test — brand/product CHƯA có rule:")
_discovery_tests = [
    "Singapore thu hồi lô sữa công thức của Nestle và Dumex vì nghi ngờ có độc tố",
    "Cocoxim Premium ra mắt dòng sản phẩm mới",
    "Heinz Ketchup giảm giá 20% toàn hệ thống",
    "10p HÚT TRONG ĐÊM",
]
for t in _discovery_tests:
    b, p, conf = discover_brand_product(t)
    print(f"  '{t[:55]}' -> brand≈'{b}' product≈'{p}' (conf={conf})")


## Cell 3c — VnRestorer: Diacritic Restoration

In [ ]:
import re as _re
from unicodedata import normalize as _unorm2

_VN_RE = _re.compile(r'[\u00C0-\u024F\u1EA0-\u1EF9]')

def _no_acc(s: str) -> str:
    s = _unorm2('NFD', s)
    s = _re.sub(r'[\u0300-\u036f]', '', s)
    return _unorm2('NFC', s.replace('đ','d').replace('Đ','D')).lower()

def _alpha_count(t): return len(_re.findall(r'[a-zA-Z\u00C0-\u024F\u1EA0-\u1EF9]', t))
def _vn_count(t):    return len(_VN_RE.findall(t))
def _vn_ratio(t):
    if not t: return 0.0
    return _vn_count(t) / max(_alpha_count(t), 1)

# ══════════════════════════════════════════════════════════════
# Domain-specific phrase dict — mined từ scandal "Đồ Hộp Hạ Long"
# Sắp theo độ dài giảm dần khi áp dụng (greedy longest-match)
# ══════════════════════════════════════════════════════════════
_PHRASES = [
    # Long compound phrases (≥6 chars compact → match cả khi viết liền không dấu cách)
    ("doihoacitnhattu30namdoivoi", "đề nghị tù hoặc ít nhất tù 30 năm đối với"),
    ("itnhattu30nam",           "ít nhất tù 30 năm"),
    ("vidung130tanthitban",     "vì dùng 130 tấn thịt bẩn"),
    ("vuangan130tanthitlonbenh","vựa ngán 130 tấn thịt lợn bệnh"),
    ("130tanthitlonbenh",       "130 tấn thịt lợn bệnh"),
    ("thudoanhopthu",           "thủ đoạn hợp thức"),
    ("choagaysoc",              "choáng gây sốc"),
    ("tieuhuysanphamchebientuthitban", "tiêu hủy sản phẩm chế biến từ thịt bẩn"),
    ("sanphamchebientuthitban", "sản phẩm chế biến từ thịt bẩn"),
    ("chebientuthitban",        "chế biến từ thịt bẩn"),
    ("loanphaipatechebien",     "loạn phải pate chế biến"),
    ("patechebien",             "pate chế biến"),
    ("tuthitlonnhiembenh",      "từ thịt lợn nhiễm bệnh"),
    ("thitlonnhiembenh",        "thịt lợn nhiễm bệnh"),
    ("thitlonbenh",             "thịt lợn bệnh"),
    ("moinguytiemankhi",        "mối nguy tiềm ẩn khi"),
    ("bankhoekhong",            "bạn khỏe không"),
    ("suthatvethitthu",         "sự thật về thịt thối"),
    ("doilotdacsan",            "đội lốt đặc sản"),
    ("gocnhinnguoidan",         "góc nhìn người dân"),
    ("gocnhin",                 "góc nhìn"),
    ("nguoidan",                "người dân"),
    ("congdongmangdangquantamnhat", "cộng đồng mạng đang quan tâm nhất"),
    ("congdongmang",            "cộng đồng mạng"),
    ("dangquantam",              "đang quan tâm"),
    ("quantamnhat",              "quan tâm nhất"),
    ("noicjngdongmang",          "nơi cộng đồng mạng"),

    ("batkhancaptongiamdocvanhanviencongty", "bắt khẩn cấp tổng giám đốc và nhân viên công ty"),
    ("batkhancaptgdvanhanviencongty", "bắt khẩn cấp TGĐ và nhân viên công ty"),
    ("batkhancaptong",           "bắt khẩn cấp tổng"),
    ("batkhancap",                "bắt khẩn cấp"),
    ("tonggiamdoc",               "tổng giám đốc"),
    ("vanhanvien",                "và nhân viên"),
    ("phechuan",                  "phê chuẩn"),
    ("vksndtphaiphong",           "VKSND TP Hải Phòng"),

    # Brand name (Đồ Hộp Hạ Long / Canfoco) KHÔNG map ở đây nữa —
    # BRAND_RULES (Cell 3) đã nhận diện brand độc lập, không cần dấu.
    # VnRestorer đụng vào brand string từng gây lỗi ghi đè (xem lịch sử sửa).
    ("congtycp",                  "Công ty CP"),

    ("banhmypatecotden",          "bánh mì pate cột đèn"),
    ("banhmypate",                "bánh mì pate"),
    ("patecotden",                "pate cột đèn"),
    ("patcotden",                 "pate cột đèn"),
    ("hatthuanchay",              "hạt thuần chay"),
    ("thuanchay",                 "thuần chay"),

    ("chauphinemphong4kho",       "châu Phi niêm phong 4 kho"),
    ("dichtachaup",               "dịch tả châu P"),

    ("highlandscoffeekhangdinhkhongsudung", "Highlands Coffee khẳng định không sử dụng"),
    ("khangdinhkhongsudung",      "khẳng định không sử dụng"),
    ("batkysanphamthit",          "bất kỳ sản phẩm thịt"),
    ("heochebiennaocuact",        "heo chế biến nào của CT"),
    ("highlandscoffeengungbantrasenvang", "Highlands Coffee ngừng bán trà sen vàng"),
    ("highlandscoffeengungban",   "Highlands Coffee ngừng bán"),
    ("ngungbantrasenvang",        "ngừng bán trà sen vàng"),
    ("trasenvang",                "trà sen vàng"),
    ("ngungban",                  "ngừng bán"),
    ("cungcap",                   "cung cấp"),
    ("codongthaimoi",             "có động thái mới"),
    ("nghivan",                   "nghi vấn"),

    ("uudaithanhto",              "ưu đãi thành tố"),
    ("banhngonmienphi",           "bánh ngon miễn phí"),
    ("mienphi",                   "miễn phí"),
    ("ketnoiniemtin",             "kết nối niềm tin"),
    ("bitancong",                 "bị tấn công"),
    ("lumcaptocgiamdoc",          "lùm xùm cấp tốc giám đốc"),
    ("dedontet",                  "để đón tết"),

    # Generic words/phrases — chỉ match theo word-boundary (xem restore_text)
    ("tam dung san xuat", "tạm dừng sản xuất"),
    ("san xuat",          "sản xuất"),
    ("san pham",          "sản phẩm"),
    ("thu hoi",           "thu hồi"),
    ("nguoi dung",        "người dùng"),
    ("nguoi tieu dung",   "người tiêu dùng"),
    ("khach hang",        "khách hàng"),
    ("cong ty",           "công ty"),
    ("co phan",           "cổ phần"),
    ("phat hien",         "phát hiện"),
    ("bi bat",            "bị bắt"),
    ("vu an",             "vụ án"),
    ("kiem tra",          "kiểm tra"),
    ("chat luong",        "chất lượng"),
    ("bao bi",            "bao bì"),
    ("nha san xuat",      "nhà sản xuất"),
    ("hang hoa",          "hàng hóa"),
    ("thi truong",        "thị trường"),
    ("an toan",           "an toàn"),
    ("suc khoe",          "sức khỏe"),
    ("dinh duong",        "dinh dưỡng"),
    ("tre em",            "trẻ em"),
    ("sua bot",           "sữa bột"),
    ("sua tuoi",          "sữa tươi"),
    ("thuc pham",         "thực phẩm"),
    ("nuoc mam",          "nước mắm"),
    ("banh mi",           "bánh mì"),
    ("ca phe",            "cà phê"),
    ("viet nam",          "Việt Nam"),
    ("ha noi",            "Hà Nội"),
    ("ho chi minh",       "Hồ Chí Minh"),
    ("da nang",           "Đà Nẵng"),
    ("ngay dang",         "ngày đăng"),
    ("nha hang",          "nhà hàng"),
    ("dia chi",           "địa chỉ"),
    ("gia re",            "giá rẻ"),
    ("chinh sach",        "chính sách"),
    ("doi voi",           "đối với"),
    ("hoac",              "hoặc"),
]
_PHRASES.sort(key=lambda kv: len(kv[0]), reverse=True)

# Bổ sung phrase mined từ train_labels (multi-word có dấu, không chứa brand keyword)
def _augment_from_train(labels_df):
    extra = []
    if labels_df is None: return extra
    seen = {k for k,_ in _PHRASES}
    for col in ('ocr_text','product_name'):
        for raw in labels_df[col].astype(str).dropna():
            raw = raw.strip()
            if len(raw) < 6: continue
            if not _re.search(r'[\u00C0-\u024F\u1EA0-\u1EF9]', raw): continue
            words = raw.split()
            for n in (3,2):
                for i in range(len(words)-n+1):
                    phrase = ' '.join(words[i:i+n])
                    if not _re.search(r'[\u00C0-\u024F\u1EA0-\u1EF9]', phrase):
                        continue
                    key = _no_acc(phrase)
                    # Loại trừ HAI CHIỀU: brand-key nằm trong phrase, HOẶC phrase
                    # (compact) là substring của 1 brand string đã biết — vd
                    # "long can" lọt lưới vì không chứa "halong"/"canfoco" trọn vẹn,
                    # nhưng "longcan" lại là substring của "halongcanfoco".
                    BRAND_KEYS = ("canfoco","canfood","ha long","halong","nan ",
                                  "milo","vinamilk","aptamil","highlands","vissan",
                                  "do hop","dohop","hop ha","hopha","pate",
                                  "nestle","dutch lady","th true","hipp","friso")
                    BRAND_STRINGS = ("halongcanfoco","dohophalong","congtydohophalong",
                                  "vinamilk","nestle","aptamil","highlands","vissan",
                                  "dutchlady","thtrue","hipp","friso","nan")
                    key_compact = key.replace(" ", "")
                    if any(b in key for b in BRAND_KEYS):
                        continue
                    if any(key_compact in bs or bs in key_compact for bs in BRAND_STRINGS if len(key_compact) >= 4):
                        continue
                    if len(key) >= 6 and key not in seen:
                        seen.add(key)
                        extra.append((key, phrase))
    return extra

_PHRASES = _PHRASES + _augment_from_train(train_labels_df)
_PHRASES.sort(key=lambda kv: len(kv[0]), reverse=True)
print(f"VnRestorer: {len(_PHRASES):,} phrases (static + mined from train_labels)")

# ══════════════════════════════════════════════════════════════
# FIX HIỆU NĂNG: compile TẤT CẢ regex MỘT LẦN DUY NHẤT ở đây,
# thay vì compile lại mỗi lần restore_text() được gọi (= mỗi ảnh).
# Đây là nguyên nhân chính khiến vòng lặp OCR bị chậm.
# ══════════════════════════════════════════════════════════════
_COMPILED_PHRASES = []
for _src, _tgt in _PHRASES:
    if len(_src) < 3:
        continue
    _src_compact = _src.replace(' ', '')
    if len(_src_compact) >= 6:
        for _pat in {_src, _src_compact}:
            _COMPILED_PHRASES.append(
                (_re.compile(_re.escape(_pat), _re.IGNORECASE), _tgt)
            )
    else:
        _regex = _re.compile(
            r'(?<![\w\u00C0-\u024F\u1EA0-\u1EF9])' + _re.escape(_src) +
            r'(?![\w\u00C0-\u024F\u1EA0-\u1EF9])', _re.IGNORECASE)
        _COMPILED_PHRASES.append((_regex, _tgt))

print(f"VnRestorer: {len(_COMPILED_PHRASES):,} compiled patterns ready")


def restore_text(text: str) -> str:
    """Phục hồi dấu tiếng Việt. Chỉ activate khi vn_ratio < 0.10.

    Dùng list ký tự + mask "protected" để các phrase match SAU không được
    ghi đè lên vùng text đã được 1 phrase dài hơn sửa đúng trước đó
    (tránh lỗi kiểu "Long Canfoco" bị phrase ngắn "long can" mine từ
    train_labels phá hỏng thành "LONG CẦNfoco").
    """
    if not text: return text
    if _vn_ratio(text) >= 0.10:
        return text

    chars = list(text)
    protected = [False] * len(chars)

    def _apply(regex, tgt):
        nonlocal chars, protected
        result = ''.join(chars)
        out_chars, out_protected = [], []
        pos = 0
        for m in regex.finditer(result):
            st, en = m.start(), m.end()
            if any(protected[st:en]):
                continue  # vùng này đã được phrase dài hơn sửa trước đó → bỏ qua
            out_chars.append(result[pos:st])
            out_protected.extend(protected[pos:st])
            out_chars.append(tgt)
            out_protected.extend([True] * len(tgt))
            pos = en
        out_chars.append(result[pos:])
        out_protected.extend(protected[pos:])
        chars = list(''.join(out_chars))
        protected = out_protected

    # Dùng regex ĐÃ COMPILE SẴN ở trên — không compile lại nữa
    for regex, tgt in _COMPILED_PHRASES:
        _apply(regex, tgt)

    return ''.join(chars)


# Test
print("\nVnRestorer test:")
for txt in [
    "tam dung san xuat",
    "GOCNHINO NGUOIDAN PATECOTDEN Hạ Long SUTHATVETHITTHU DOILOTDACSAN",
    "Nestlé NAN OPTIpro PLUS",   # đã có dấu → giữ nguyên
    "HALONGCANFOCO pate",        # brand → giữ nguyên (vn_ratio thấp nhưng PHRASES không inject brand)
]:
    r = restore_text(txt)
    print(f"  '{txt[:60]}'")
    print(f"  → '{r[:60]}'")

VnRestorer: 29,638 phrases (static + mined from train_labels)
VnRestorer: 57,324 compiled patterns ready

VnRestorer test:
  'tam dung san xuat'
  → 'tạm dừng sản xuất'
  'GOCNHINO NGUOIDAN PATECOTDEN Hạ Long SUTHATVETHITTHU DOILOTD'
  → 'GÓC NHÌNO NGƯỜI DÂN pate cột đèn Hạ Long sự thật về thịt thố'
  'Nestlé NAN OPTIpro PLUS'
  → 'Nestlé NAN OPTIpro PLUS'
  'HALONGCANFOCO pate'
  → 'HALONGCANFOCO pate'


## Cell 4 — OCR Engine: PaddleOCR (chính) + HybridOCR PaddleDet+VietOCR (fallback)


In [ ]:
import time
import numpy as np
from paddleocr import PaddleOCR

# ══════════════════════════════════════════════════════════════
# TẦNG 1 — PaddleOCR full pipeline (API cũ <3.0, giữ nguyên cấu
# hình gốc của notebook để không phá vỡ pipeline đã chạy ổn).
# ══════════════════════════════════════════════════════════════
ocr_engine = PaddleOCR(
    ocr_version="PP-OCRv4",
    use_angle_cls=True,
    lang="vi",
    use_gpu=False,
    show_log=False,
    det_db_score_mode="slow",
    rec_batch_num=8,
)
print("PaddleOCR: PP-OCRv4 lang=vi ✓")

# ══════════════════════════════════════════════════════════════
# TẦNG 2 (fallback) — HybridOCR: dùng LẠI ocr_engine ở trên cho
# detection-only (det=True, rec=False, cls=False — API cũ, không
# cần khởi tạo model detection riêng) + VietOCR vgg_seq2seq cho
# recognition (nhanh ~12ms/box, tối ưu dấu tiếng Việt hơn Paddle
# rec gốc). Chỉ load VietOCR khi thực sự cần (lazy) để không tốn
# thời gian/RAM nếu tầng 1 đã đủ tốt cho hầu hết ảnh.
# ══════════════════════════════════════════════════════════════
_vietocr_predictor = None

def _get_vietocr_predictor():
    global _vietocr_predictor
    if _vietocr_predictor is None:
        from vietocr.tool.predictor import Predictor
        from vietocr.tool.config import Cfg
        config = Cfg.load_config_from_name("vgg_seq2seq")
        config["device"] = "cpu"
        config["predictor"]["beamsearch"] = False
        _vietocr_predictor = Predictor(config)
        print("  [lazy-init] VietOCR vgg_seq2seq ✓")
    return _vietocr_predictor


def load_and_prep(image_id: str, max_dim=1280, min_dim=480):
    path = IMAGES_DIR / f"{image_id}.jpg"
    if not path.exists(): return None
    try:
        img = Image.open(path).convert("RGB")
    except Exception: return None
    w, h = img.size
    if max(w, h) < min_dim:
        s = min_dim / max(w, h)
        img = img.resize((int(w*s), int(h*s)), Image.LANCZOS)
    elif max(w, h) > max_dim:
        s = max_dim / max(w, h)
        img = img.resize((int(w*s), int(h*s)), Image.LANCZOS)
    img = ImageEnhance.Sharpness(img).enhance(1.8)
    img = ImageEnhance.Contrast(img).enhance(1.35)
    return img

_OCR_FIXES = [
    (r'\b[Vv]inam[i1l][l1]k\b',    'Vinamilk'),
    (r'\b[Cc]anf[uo][ck][oa]\b',   'Canfoco'),
    (r'\bTH.?Tru[e3]\b',           'TH True'),
    (r'\b[Nn]estl[e3é]\b',         'Nestlé'),
    (r'\b[Mm][i1]lo\b',            'Milo'),
    (r'\b[Hh]ighland[s]?\b',       'Highlands'),
    (r'\b[Hh][aà].?[Ll][o0]ng\b',  'Hạ Long'),
    (r'\b[Nn][Aa][Nn]\b',          'NAN'),
    (r'\b[Aa]ptam[i1]l\b',         'Aptamil'),
    (r'\b[Hh][Ii][Pp][Pp]\b',      'HiPP'),
    (r'\b[Vv]issan\b',             'Vissan'),
    (r'(?<=[a-z])0(?=[a-z])',        'o'),
]

def _fix(text: str) -> str:
    for pat, rep in _OCR_FIXES:
        text = re.sub(pat, rep, text, flags=re.IGNORECASE)
    return text

def _dedup(text: str) -> str:
    toks = text.split()
    if not toks: return ''
    out = [toks[0]]
    for t in toks[1:]:
        if t.lower() != out[-1].lower(): out.append(t)
    return ' '.join(out)

def postprocess_ocr(text: str) -> str:
    text = re.sub(r'[\n\t\r]', ' ', text)
    text = re.sub(r'[^\w\s\u00C0-\u024F\u1EA0-\u1EF9.,;:!?()/%%\-@#]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    text = _fix(text)
    text = restore_text(text)
    text = _dedup(text)
    return text[:500]

def _parse(result, thresh=0.30):
    """Parse output của ocr_engine.ocr(img, cls=True) — API cũ trả về
    [[ [box, (text, score)], ... ]]."""
    lines = []
    if not result or not result[0]: return lines
    for line in result[0]:
        if not line or len(line) < 2: continue
        ti = line[1]
        if isinstance(ti, (list, tuple)) and len(ti) >= 2:
            txt, score = str(ti[0]).strip(), float(ti[1])
            if score >= thresh and txt:
                lines.append((txt, score))
    return lines

def _is_noise(raw: str) -> bool:
    if not raw: return True
    toks = raw.split()
    if len(toks) <= 1 and len(raw) < 4: return True
    alpha = sum(1 for t in toks if re.search(r'[a-zA-Z\u00C0-\u024F\u1EA0-\u1EF9]', t))
    return alpha / max(len(toks), 1) < 0.20


def _run_paddle_full(img) -> tuple:
    """Tầng 1: PaddleOCR full pipeline (API cũ .ocr(cls=True)).
    Trả về (raw_text, mean_score)."""
    try:
        result = ocr_engine.ocr(np.array(img), cls=True)
    except Exception:
        return '', 0.0
    pairs = _parse(result, thresh=0.30)
    if not pairs: return '', 0.0
    raw = ' '.join(t for t, _ in pairs)
    mean_score = sum(s for _, s in pairs) / len(pairs)
    return raw, mean_score


def _crop_quad(img_np, box):
    """Cắt vùng chữ theo box 4 điểm [[x,y],[x,y],[x,y],[x,y]]
    (axis-aligned bbox, đủ dùng cho text ngang/nghiêng nhẹ)."""
    pts = np.array(box, dtype=np.int32)
    x0, x1 = max(0, pts[:,0].min()), min(img_np.shape[1], pts[:,0].max())
    y0, y1 = max(0, pts[:,1].min()), min(img_np.shape[0], pts[:,1].max())
    if x1 <= x0 or y1 <= y0: return None
    return img_np[y0:y1, x0:x1]


def _run_hybrid(img) -> str:
    """Tầng 2 (fallback): detection-only bằng CHÍNH ocr_engine đã có
    (det=True, rec=False, cls=False — API cũ, không tốn thêm RAM
    load model detection riêng) + VietOCR vgg_seq2seq để nhận dạng
    từng vùng chữ. Dùng khi tầng 1 cho kết quả rỗng/yếu."""
    img_np = np.array(img)
    try:
        det_result = ocr_engine.ocr(img_np, det=True, rec=False, cls=False)
    except Exception:
        return ''
    if not det_result or not det_result[0]: return ''
    boxes = det_result[0]  # list of 4-point boxes

    predictor = _get_vietocr_predictor()
    texts = []
    for box in boxes:
        crop = _crop_quad(img_np, box)
        if crop is None or crop.size == 0: continue
        try:
            txt = predictor.predict(Image.fromarray(crop))
        except Exception:
            continue
        if txt and txt.strip():
            texts.append(txt.strip())
    return ' '.join(texts)


def run_ocr(image_id: str, weak_score_thresh: float = 0.55) -> tuple:
    """Trả về (ocr_text, brand_name, product_name).

    Chiến lược 2 tầng:
      1) PaddleOCR full pipeline (nhanh) — dùng làm kết quả chính.
      2) Nếu (1) rỗng HOẶC mean confidence < weak_score_thresh →
         fallback sang HybridOCR (detection-only bằng ocr_engine +
         VietOCR vgg_seq2seq recognition), vốn nhận dạng tiếng Việt
         tốt hơn nhưng chậm hơn — chỉ chạy khi thật sự cần.
    """
    img = load_and_prep(image_id)
    if img is None: return '', '', ''

    raw, score = _run_paddle_full(img)

    if not raw or score < weak_score_thresh:
        hybrid_raw = _run_hybrid(img)
        if hybrid_raw and not _is_noise(hybrid_raw):
            if not raw or len(hybrid_raw) >= len(raw):
                raw = hybrid_raw

    if _is_noise(raw):
        return '', '', ''

    ocr_text = postprocess_ocr(raw)
    brand_name, product_name = predict_brand_product(ocr_text)
    return ocr_text, brand_name, product_name


print("\nSmoke test 5 ảnh đầu:")
for img_id in test_df['image_id'].iloc[:5]:
    t0 = time.time()
    ocr, brand, prod = run_ocr(img_id)
    vn = len(_VN_RE.findall(ocr))
    status = '✅' if ocr else '⚠️ EMPTY'
    print(f"  {status} {img_id} ({time.time()-t0:.1f}s) | VN={vn}")
    if ocr:
        print(f"    ocr    : {ocr[:90]}")
        print(f"    brand  : {brand or '(empty)'}")
        print(f"    product: {prod or '(empty)'}")


## Cell 5 — Main Loop

In [ ]:
import os

# Bỏ comment để chạy lại từ đầu:
# if CHECKPOINT_CSV.exists(): os.remove(CHECKPOINT_CSV)

SAVE_EVERY = 25
done_ids, results = set(), []

if CHECKPOINT_CSV.exists():
    ckpt     = pd.read_csv(CHECKPOINT_CSV, keep_default_na=False)
    done_ids = set(ckpt["image_id"])
    results  = ckpt.to_dict("records")
    print(f"Tiếp tục: {len(done_ids):,} đã xong")
else:
    print("Bắt đầu mới")

pending = [i for i in test_df["image_id"] if i not in done_ids]
print(f"Còn lại: {len(pending):,} | Đã xong: {len(done_ids):,}")

ocr_empty = 0
for idx, img_id in enumerate(tqdm(pending, desc="OCR")):
    ocr_text, brand_name, product_name = run_ocr(img_id)
    if not ocr_text and (IMAGES_DIR / f"{img_id}.jpg").exists():
        ocr_empty += 1
    results.append({
        "image_id": img_id,
        "ocr_text": ocr_text,
        "brand_name": brand_name,
        "product_name": product_name,
    })
    if (idx+1) % SAVE_EVERY == 0:
        pd.DataFrame(results).to_csv(CHECKPOINT_CSV, index=False, encoding="utf-8")
    if (idx+1) % 250 == 0:
        df_tmp = pd.DataFrame(results)
        of = (df_tmp["ocr_text"].str.strip()!="").mean()
        bf = (df_tmp["brand_name"].str.strip()!="").mean()
        pf = (df_tmp["product_name"].str.strip()!="").mean()
        print(f"[{idx+1}] {img_id} | OCR fill {of:.0%} | brand fill {bf:.0%} | product fill {pf:.0%}")

pd.DataFrame(results).to_csv(CHECKPOINT_CSV, index=False, encoding="utf-8")
df_r = pd.DataFrame(results)
print(f"\nĐã xử lý    : {len(df_r):,}")
print(f"OCR fill    : {(df_r['ocr_text'].str.strip()!='').mean():.1%}  ← nên ≥ 70%")
print(f"Brand fill  : {(df_r['brand_name'].str.strip()!='').mean():.1%}")
print(f"Product fill: {(df_r['product_name'].str.strip()!='').mean():.1%}")
print(f"OCR trống   : {ocr_empty:,}  ← chấp nhận được nếu ảnh thật sự không có chữ")

print()
print_discovery_cache_summary(min_count=2)


## Cell 6 — Export submission.csv

In [ ]:
import csv

sub    = pd.DataFrame(results)[["image_id","ocr_text","brand_name","product_name"]]
sample = pd.read_csv(SAMPLE_CSV, keep_default_na=False)
exp    = set(sample["image_id"]); got = set(sub["image_id"])

checks = {
    "Row count"    : len(sub)==len(sample),
    "No extra IDs" : len(got-exp)==0,
    "No missing"   : len(exp-got)==0,
    "No duplicates": not sub["image_id"].duplicated().any(),
    "No nulls"     : not sub.isnull().any().any(),
    "Columns OK"   : list(sub.columns)==["image_id","ocr_text","brand_name","product_name"],
}
all_pass = all(checks.values())
for name, ok in checks.items():
    print(f"  [{'PASS' if ok else 'FAIL'}] {name}")

if all_pass:
    sub = sub.set_index("image_id").reindex(sample["image_id"]).reset_index()
    for col in ("ocr_text","brand_name","product_name"):
        sub[col] = sub[col].fillna("").astype(str).str.strip()
    out = sub.copy()
    for col in ("ocr_text","brand_name","product_name"):
        out.loc[out[col]=="", col] = " "
    out.to_csv(OUTPUT_CSV, index=False, encoding="utf-8", quoting=csv.QUOTE_ALL)
    print(f"\nSaved: {OUTPUT_CSV}")
    print(f"OCR fill    : {(sub['ocr_text'].str.strip()!='').mean():.1%}")
    print(f"Brand fill  : {(sub['brand_name'].str.strip()!='').mean():.1%}")
    print(f"Product fill: {(sub['product_name'].str.strip()!='').mean():.1%}")
    display(sub.head())
else:
    print("\nFAIL — sửa lỗi trước khi submit")


## Cell 6b — Batch Mining: gợi ý Brand/Product MỚI (review tay)


In [ ]:
from collections import Counter

# ══════════════════════════════════════════════════════════════
# BATCH MINING — gợi ý brand/product MỚI để bổ sung tay vào
# BRAND_RULES / LINE_RULES (Cell 3). Chạy SAU khi đã có `results`
# từ Cell 5 (Main Loop). Không tự động thêm vào rules — chỉ xuất
# bảng gợi ý kèm tần suất + ví dụ ocr_text để bạn review.
# ══════════════════════════════════════════════════════════════

MIN_FREQ = 3          # cụm phải xuất hiện ≥ N ảnh mới đáng xem xét
MIN_CONFIDENCE = 0.45  # khớp với ngưỡng discover_brand_product()

df_results = pd.DataFrame(results)

# --- (A) Brand đã có nhưng KHÔNG match rule, KHÔNG match ML, ---
# --- chỉ được Discovery Layer (tầng 4, heuristic+NER) đoán ra ---
discovery_hits = []
for _, row in df_results.iterrows():
    ocr = str(row.get("ocr_text", "")).strip()
    if not ocr: continue

    rule_brand, _ = extract_brand_product(ocr)
    if rule_brand: continue   # đã có rule match -> không tính là "mới"

    d_brand, d_prod, conf = discover_brand_product(ocr)
    if d_brand and conf >= MIN_CONFIDENCE:
        discovery_hits.append({
            "image_id": row["image_id"],
            "brand_guess": d_brand,
            "product_guess": d_prod,
            "confidence": conf,
            "ocr_text": ocr[:80],
        })

discovery_df = pd.DataFrame(discovery_hits)

print("═" * 70)
print("GỢI Ý BRAND/PRODUCT MỚI (phát hiện bởi Discovery Layer)")
print("═" * 70)

if len(discovery_df) == 0:
    print("Không có brand mới nào được Discovery Layer phát hiện trong batch này.")
else:
    # Gộp theo brand_guess, đếm tần suất + lấy ví dụ
    agg = (discovery_df.groupby("brand_guess")
           .agg(freq=("image_id", "count"),
                avg_confidence=("confidence", "mean"),
                example_ocr=("ocr_text", "first"),
                example_image=("image_id", "first"))
           .reset_index()
           .sort_values("freq", ascending=False))

    suggestions = agg[agg["freq"] >= MIN_FREQ]
    rare = agg[agg["freq"] < MIN_FREQ]

    print(f"\n>> ĐÁNG XEM XÉT (xuất hiện ≥{MIN_FREQ} lần) — {len(suggestions)} brand:\n")
    if len(suggestions):
        for _, r in suggestions.iterrows():
            print(f"  '{r['brand_guess']}'  (x{r['freq']}, conf TB={r['avg_confidence']:.2f})")
            print(f"      vd ảnh {r['example_image']}: \"{r['example_ocr']}\"")
    else:
        print("  (không có brand nào đạt ngưỡng tần suất)")

    print(f"\n>> HIẾM GẶP (<{MIN_FREQ} lần, có thể là noise, xem thêm trước khi thêm rule) — {len(rare)} brand:\n")
    if len(rare):
        for _, r in rare.head(15).iterrows():
            print(f"  '{r['brand_guess']}'  (x{r['freq']}, conf={r['avg_confidence']:.2f})")

    print(f"\n>> Tổng cộng {len(discovery_df)} ảnh được gán brand/product bởi Discovery Layer "
          f"({len(discovery_df)/max(len(df_results),1)*100:.1f}% batch).")

    print("\n--- Cách bổ sung vào BRAND_RULES (Cell 3) ---")
    print("Nếu thấy brand hợp lý ở trên, thêm dòng vào BRAND_RULES, ví dụ:")
    print("    (r're?gex_pattern_brand', 'Tên Brand Canonical'),")
    print("rồi chạy lại Cell 3 + Cell 3b để áp dụng cho lần inference sau.")


# --- (B) Cụm từ viết hoa lặp lại nhiều lần nhưng KHÔNG match ---
# --- rule LẪN không được Discovery Layer chọn (do dưới ngưỡng   ---
# --- confidence) — vẫn đáng xem vì có thể là brand bị OCR yếu   ---
print("\n" + "═" * 70)
print("CỤM TỪ LẶP LẠI NHIỀU (chưa match rule, dưới ngưỡng Discovery)")
print("═" * 70)

leftover_counter = Counter()
leftover_example = {}
for _, row in df_results.iterrows():
    ocr = str(row.get("ocr_text", "")).strip()
    if not ocr: continue
    rule_brand, _ = extract_brand_product(ocr)
    if rule_brand: continue

    for run_text, start_pos, end_pos in _extract_capitalized_runs(_tokenize_with_spans(ocr)):
        sc = _score_candidate(run_text, ocr, start_pos)
        if sc < MIN_CONFIDENCE:   # đã bị Discovery Layer bỏ qua ở trên
            key = run_text.strip()
            if len(key) >= 3:
                leftover_counter[key] += 1
                leftover_example.setdefault(key, (row["image_id"], ocr[:80]))

top_leftover = [(k, v) for k, v in leftover_counter.most_common(20) if v >= MIN_FREQ]
if top_leftover:
    print(f"\n{len(top_leftover)} cụm xuất hiện ≥{MIN_FREQ} lần (review thủ công, độ tin cậy thấp hơn phần A):\n")
    for cand, freq in top_leftover:
        img_id, ex = leftover_example[cand]
        print(f"  '{cand}'  (x{freq})  vd ảnh {img_id}: \"{ex}\"")
else:
    print("\nKhông có cụm nào đủ tần suất để gợi ý thêm.")

print("\n(Bảng này CHỈ để tham khảo — không tự động sửa rules."
      " Hãy review từng dòng trước khi thêm vào BRAND_RULES/LINE_RULES.)")

print("\n" + "═" * 70)
print("ONLINE CACHE — brand đã \"học\" được trong lúc chạy batch này")
print("═" * 70)
print_discovery_cache_summary(min_count=1)
print("\n(Cache chỉ sống trong RAM của session hiện tại — sẽ mất khi"
      " restart kernel. Brand nào thấy ổn ở đây nên được thêm CỐ ĐỊNH"
      " vào BRAND_RULES ở Cell 3 để không phải học lại mỗi lần chạy.)")


## Cell 7 — Local Score (Score = 0.4×F1_brand + 0.35×(1−CER) + 0.25×F1_product)


In [ ]:
def composite_score(sol, sub, id_col="image_id"):
    """Điểm theo format vòng mới (BTC):
        Score = 0.4 × F1_brand + 0.35 × (1 − CER) + 0.25 × F1_product

    - F1_brand   : F1 theo token, so khớp brand_name (GT vs pred)
    - CER        : Character Error Rate trên ocr_text (Levenshtein / len(gt))
    - F1_product : F1 theo token, so khớp product_name (GT vs pred)

    Case đặc biệt (áp dụng riêng cho TỪNG thành phần, không chỉ tổng):
      - GT rỗng & pred rỗng  → thành phần đó = 1.0 (không bị phạt)
      - GT có nhãn & pred rỗng (hoặc ngược lại) → thành phần đó = 0.0
    """
    def clean(v): return "" if pd.isna(v) else str(v).strip()

    m = sol.merge(sub, on=id_col, suffixes=("_gt", "_pred"))

    def f1(gt, pred):
        gt, pred = clean(gt), clean(pred)
        if not gt and not pred: return 1.0          # Case D: cả hai rỗng → full điểm
        g, p = set(gt.lower().split()), set(pred.lower().split())
        if not g or not p: return 0.0                # Case C: 1 bên rỗng → 0
        c = g & p
        pr, rc = len(c) / len(p), len(c) / len(g)
        return 0.0 if pr + rc == 0 else 2 * pr * rc / (pr + rc)

    def cer(gt, pred):
        gt, pred = clean(gt), clean(pred)
        if not gt and not pred: return 0.0           # Case D: cả hai rỗng → CER=0 → (1-CER)=1.0
        if not gt: return 1.0                        # GT rỗng nhưng pred không rỗng → CER tối đa
        if not pred: return 1.0                       # Case C: pred rỗng khi GT có nhãn → CER tối đa
        m2, n = len(gt), len(pred)
        dp = list(range(n + 1))
        for i in range(1, m2 + 1):
            prev, dp[0] = dp[0], i
            for j in range(1, n + 1):
                tmp = dp[j]
                dp[j] = prev if gt[i-1] == pred[j-1] else 1 + min(prev, dp[j], dp[j-1])
                prev = tmp
        return min(dp[n] / len(gt), 1.0)

    f1_brand   = m.apply(lambda r: f1( r["brand_name_gt"],   r["brand_name_pred"]),   axis=1)
    f1_product = m.apply(lambda r: f1( r["product_name_gt"], r["product_name_pred"]), axis=1)
    cers       = m.apply(lambda r: cer(r["ocr_text_gt"],     r["ocr_text_pred"]),     axis=1)

    per_image = 0.4 * f1_brand + 0.35 * (1 - cers) + 0.25 * f1_product

    print(f"  F1_brand   (mean) : {f1_brand.mean():.4f}")
    print(f"  1-CER      (mean) : {(1 - cers).mean():.4f}")
    print(f"  F1_product (mean) : {f1_product.mean():.4f}")

    return round(per_image.mean(), 4)


# ── Self-test theo 4 case nêu trong đề bài ──
def _t(gt_brand, gt_ocr, gt_prod, pr_brand, pr_ocr, pr_prod):
    sol  = pd.DataFrame([{"image_id":"x","ocr_text":gt_ocr,"brand_name":gt_brand,"product_name":gt_prod}])
    pred = pd.DataFrame([{"image_id":"x","ocr_text":pr_ocr,"brand_name":pr_brand,"product_name":pr_prod}])
    return composite_score(sol, pred)

print("Case A (đúng hết):")
print("  Score =", _t("Dove","Dove Smoothie tẩy da chết","Smoothie tẩy da chết",
                       "Dove","Dove Smoothie tẩy da chết","Smoothie tẩy da chết"))
print("\nCase B (brand đúng, product/OCR thiếu từ):")
print("  Score =", _t("Dove","Dove Smoothie tẩy da chết","Smoothie tẩy da chết",
                       "Dove","Dove Smoothie","Smoothie"))
print("\nCase C (submit rỗng khi GT có nhãn):")
print("  Score =", _t("Dove","Dove Smoothie tẩy da chết","Smoothie tẩy da chết",
                       " "," "," "))
print("\nCase D (cả GT và pred đều rỗng):")
print("  Score =", _t(""," ","", " ", " ", " "))


for sp in [Path("smce_dataset/solution.csv"), Path("/kaggle/input/smce-solution/solution.csv")]:
    if sp.exists():
        sol  = pd.read_csv(sp, keep_default_na=False)
        pred = pd.DataFrame(results)[["image_id","ocr_text","brand_name","product_name"]]
        print(f"\nLocal Score: {composite_score(sol, pred):.4f}")
        break
else:
    print("\nsolution.csv không có — bình thường với participant")
